# Ecowitt — schema explorer and hourly metric table

Samples every table and view **discovered at runtime** from the catalog, then
builds an hour × metric table for yesterday and today.

**Prerequisite:** the SSH tunnel must be open, in a separate terminal:

```bash
./infra/tunnel.sh
```

Read-only throughout — the `ecowitt_ro` role cannot write even by accident.

In [2]:
# Preflight: make this notebook work whatever kernel it lands on.
#
# VS Code writes the selected kernel back INTO this file, so pinning a
# kernelspec here loses to whatever was last used. This machine also has
# several Pythons (Anaconda, other project venvs). Rather than fight that,
# fall back to the project venv's site-packages when the current interpreter
# lacks the dependencies -- safe only when the Python versions match exactly,
# because compiled extensions like psycopg are ABI-specific.
import sys
from importlib.util import find_spec
from pathlib import Path

NEEDED = ("psycopg", "sqlalchemy", "pandas")


def _repo_venv_site_packages():
    """Walk up from the notebook looking for a sibling .venv."""
    for base in (Path.cwd(), *Path.cwd().parents):
        sp = base / ".venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
        if sp.is_dir():
            return sp
    return None


missing = [m for m in NEEDED if find_spec(m) is None]
if missing:
    sp = _repo_venv_site_packages()
    if sp and str(sp) not in sys.path:
        sys.path.insert(0, str(sp))
        missing = [m for m in NEEDED if find_spec(m) is None]
        if not missing:
            print(f"NOTE: running on {sys.executable}")
            print(f"      which lacks {', '.join(NEEDED)}, so this notebook added")
            print(f"      {sp}")
            print("      to sys.path as a fallback. It works because both interpreters are")
            print(f"      Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}.")
            print("      Cleaner: select the 'Python (ecowitt)' kernel (VS Code: kernel name,")
            print("      top-right of the notebook).")

if missing:
    raise SystemExit(
        f"Wrong kernel, and the fallback could not help.\n\n"
        f"  running on : {sys.executable}\n"
        f"  missing    : {', '.join(missing)}\n\n"
        "Select the 'Python (ecowitt)' kernel.\n"
        "  VS Code    : click the kernel name in the notebook's TOP-RIGHT corner\n"
        "               -> Select Another Kernel... -> Jupyter Kernel...\n"
        "  JupyterLab : Kernel > Change Kernel\n\n"
        "If it is not offered, register it once from the repo root:\n"
        "    .venv/bin/python -m ipykernel install --user --name ecowitt \\\n"
        "        --display-name 'Python (ecowitt)'"
    )

print(f"kernel OK: {sys.executable}")


SystemExit: Wrong kernel.

  running on : /Users/marcalexander/projects/evidence_project/.venv/bin/python
  missing    : psycopg
  want       : <repo>/.venv/bin/python

VS Code:     click the kernel name in the notebook's TOP-RIGHT corner
             -> Select Another Kernel... -> Jupyter Kernel...
             -> 'Python (ecowitt)'
             (or Python Environments... -> the .venv in ecowitt_weather)

JupyterLab:  Kernel > Change Kernel > 'Python (ecowitt)'

If 'Python (ecowitt)' is not offered, register it once from the repo root:
    .venv/bin/python -m ipykernel install --user --name ecowitt \
        --display-name 'Python (ecowitt)'
then reload the window (VS Code: Cmd-Shift-P, 'Developer: Reload Window').

In [ ]:
import os
import subprocess
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
import psycopg
from sqlalchemy import create_engine, text

HOST, PORT = '127.0.0.1', 5433
DB, USER   = 'ecowitt', 'ecowitt_ro'
SECRET     = 'ecowitt-readonly-password'

# Day boundaries are a human idea, so they are computed in console-local time.
# Storage stays UTC; only the display is shifted.
DISPLAY_TZ = ZoneInfo('America/Chicago')
SAMPLE_ROWS = 5

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 250)


def password() -> str:
    """Env var first, else Secret Manager. Never hardcoded in this notebook."""
    if pw := os.environ.get('ECOWITT_RO_PASSWORD'):
        return pw
    out = subprocess.run(
        ['gcloud', 'secrets', 'versions', 'access', 'latest', f'--secret={SECRET}'],
        capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f'could not read {SECRET}: {out.stderr.strip()[:200]}')
    return out.stdout.strip()


from urllib.parse import quote_plus

engine = create_engine(
    f'postgresql+psycopg://{USER}:{quote_plus(password())}@{HOST}:{PORT}/{DB}',
    connect_args={'connect_timeout': 10})

try:
    conn = engine.connect()
except Exception as exc:
    raise SystemExit(
        f'Cannot reach {HOST}:{PORT}.\n'
        'The tunnel is almost certainly not running. In another terminal:\n'
        '    ./infra/tunnel.sh\n'
        f'Original error: {str(exc).splitlines()[0]}') from None

print(pd.read_sql(text('select current_user, current_database(), now()'), conn).to_string(index=False))

ModuleNotFoundError: No module named 'psycopg'

## 1. Discover objects from the catalog

Nothing here is hardcoded. Every table, view, materialised view and partitioned
table in `public` is picked up at runtime, so a new object added later is
automatically in scope for the sampling below.

In [ ]:
OBJECTS_SQL = '''
SELECT c.relname                       AS object_name,
       CASE c.relkind WHEN 'r' THEN 'table'
                      WHEN 'p' THEN 'partitioned table'
                      WHEN 'v' THEN 'view'
                      WHEN 'm' THEN 'materialized view'
       END                             AS kind,
       pg_size_pretty(pg_total_relation_size(c.oid)) AS size,
       obj_description(c.oid)          AS comment
  FROM pg_class c
  JOIN pg_namespace n ON n.oid = c.relnamespace
 WHERE n.nspname = 'public'
   AND c.relkind IN ('r', 'p', 'v', 'm')
 ORDER BY c.relkind, c.relname
'''

objects = pd.read_sql(text(OBJECTS_SQL), conn)
print(f'{len(objects)} objects discovered\n')
objects

## 2. Row counts

Counted per object rather than read from `pg_class.reltuples`, which is only a
planner estimate and can be stale or plain wrong on a young table.

In [ ]:
counts = []
for row in objects.itertuples():
    try:
        n = conn.execute(text(f'SELECT count(*) FROM public."{row.object_name}"')).scalar()
        counts.append({'object_name': row.object_name, 'kind': row.kind,
                       'rows': n, 'error': None})
    except Exception as exc:
        conn.rollback()
        counts.append({'object_name': row.object_name, 'kind': row.kind,
                       'rows': None, 'error': str(exc).splitlines()[0][:80]})

pd.DataFrame(counts)

## 3. Sample rows from every object

A failure on one object is reported and skipped rather than aborting the batch —
an empty table and a broken view should be distinguishable at a glance.

In [ ]:
for row in objects.itertuples():
    label = f'{row.object_name}  ({row.kind})'
    print('=' * 100)
    print(label)
    print('=' * 100)
    try:
        df = pd.read_sql(
            text(f'SELECT * FROM public."{row.object_name}" LIMIT {SAMPLE_ROWS}'), conn)
    except Exception as exc:
        conn.rollback()
        print(f'  ERROR: {str(exc).splitlines()[0][:120]}\n')
        continue
    if df.empty:
        print('  (no rows)\n')
    else:
        display(df)
    print()

## 4. Hourly metric table — yesterday + today

Rows are hours, columns are metrics.

**Each metric is aggregated by its own rule from `metric_catalog`, not by a
blanket average.** That is not fussiness:

| kind | rule | why a plain average is wrong |
|---|---|---|
| `circular` | vector mean | averaging 350° and 10° linearly gives 180° — due south for a north wind. Measured 17 north-crossings in a single day of this station's data. |
| `extremum` | `max` | a gust is a per-interval maximum; averaging understates peak wind |
| `accumulator` | `last` | running totals that reset; averaging them is meaningless |
| `opaque` / `status` | `last` | not measurements; no arithmetic is valid |

The aggregation happens in SQL so the rules stay next to the data that defines them.

In [ ]:
HOURLY_SQL = '''
WITH bounded AS (
    SELECT o.ts_utc, o.metric, o.value_num, c.resample_rule
      FROM observation o
      JOIN metric_catalog c USING (metric)
     WHERE o.ts_utc >= :start
       AND o.ts_utc <  :end
       AND o.value_num IS NOT NULL
),
hourly AS (
    SELECT date_trunc('hour', ts_utc AT TIME ZONE :tz) AS hour_local,
           metric,
           resample_rule,
           avg(value_num)                                  AS v_mean,
           max(value_num)                                  AS v_max,
           (array_agg(value_num ORDER BY ts_utc DESC))[1]  AS v_last,
           -- vector mean: atan2 of the mean sine and mean cosine
           atan2d(avg(sind(value_num)), avg(cosd(value_num))) AS v_vector
      FROM bounded
     GROUP BY 1, 2, 3
)
SELECT hour_local,
       metric,
       CASE resample_rule
           WHEN 'max'         THEN v_max
           WHEN 'last'        THEN v_last
           WHEN 'carry'       THEN v_last
           WHEN 'vector_mean' THEN CASE WHEN v_vector < 0 THEN v_vector + 360
                                        ELSE v_vector END
           ELSE v_mean
       END AS value
  FROM hourly
 ORDER BY hour_local, metric
'''

now_local   = datetime.now(DISPLAY_TZ)
start_local = (now_local - timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)
end_local   = (now_local + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)

print(f'window: {start_local:%Y-%m-%d %H:%M %Z} -> {end_local:%Y-%m-%d %H:%M %Z}')

long = pd.read_sql(text(HOURLY_SQL), conn, params={
    'start': start_local, 'end': end_local, 'tz': str(DISPLAY_TZ)})
print(f'{len(long):,} metric-hours across {long.metric.nunique()} metrics')

In [ ]:
hourly = (long.pivot(index='hour_local', columns='metric', values='value')
              .sort_index())
hourly.index.name = f'hour ({DISPLAY_TZ})'

# Empty columns mean the metric reported nothing in the window -- worth seeing,
# not worth hiding.
print(f'{hourly.shape[0]} hours x {hourly.shape[1]} metrics')
hourly.round(2)

### Weather-only view

The full table includes battery voltages and status codes. This drops the
diagnostic and status metrics to leave the actual weather.

In [ ]:
WEATHER_ONLY = '''
SELECT metric FROM metric_catalog
 WHERE kind NOT IN ('diagnostic', 'status', 'opaque')
 ORDER BY metric
'''
weather = pd.read_sql(text(WEATHER_ONLY), conn).metric.tolist()
cols = [c for c in hourly.columns if c in weather]
hourly[cols].round(2)

### Coverage

How many 5-minute observations landed in each hour. A full hour is 12.
Anything less is a dropout — routine on this station, so this is a health
check rather than an alarm.

In [ ]:
COVERAGE_SQL = '''
SELECT date_trunc('hour', ts_utc AT TIME ZONE :tz) AS hour_local,
       count(DISTINCT ts_utc)                         AS slots,
       round(100.0 * count(DISTINCT ts_utc) / 12, 1)  AS pct_of_expected
  FROM observation
 WHERE ts_utc >= :start AND ts_utc < :end
 GROUP BY 1 ORDER BY 1
'''
cov = pd.read_sql(text(COVERAGE_SQL), conn, params={
    'start': start_local, 'end': end_local, 'tz': str(DISPLAY_TZ)})
# The first and last hours are partial by construction -- the window edge cuts
# them, and before ingestion started there is simply no data. Including them
# reports a dropout that never happened, so they are excluded from the summary
# but kept in the table.
full = cov.iloc[1:-1] if len(cov) > 2 else cov
if len(full):
    print(f'mean coverage (excluding partial edge hours): {full.pct_of_expected.mean():.1f}%')
    print(f'hours below 100%: {(full.slots < 12).sum()} of {len(full)}')
print('first/last rows are edge hours -- partial by construction, not dropouts')
cov

In [ ]:
conn.close()
engine.dispose()
print('connection closed')